In [ ]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader 
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [ ]:
def load_pdf(data):
    loader = DirectoryLoader(
        data,
        glob = "*.pdf",
        loader_cls=PyPDFLoader

    )
    documents = loader.load()
    return documents

In [ ]:
import os
os.chdir(r"c:\Users\A531927\OneDrive - Volvo Group\Desktop\Medical_chatbot\Medical-Chatbot")
executed_data=load_pdf("Data")

In [ ]:
executed_data

In [ ]:
len(executed_data)

In [ ]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document] :
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document (
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs


In [ ]:
minimal_docs = filter_to_minimal_docs(executed_data)

In [ ]:
minimal_docs

In [ ]:
#split the documents into chunks
def text_split(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
        length_function=len
    )
    text=text_splitter.split_documents(minimal_docs)
    return text



In [ ]:
text_chunk=text_split(minimal_docs)

In [ ]:
print(f"Number of chunks: {len(text_chunk)}")

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings


In [ ]:
import os
import requests

# Disable SSL verification for corporate proxy
os.environ["CURL_CA_BUNDLE"] = ""
os.environ["REQUESTS_CA_BUNDLE"] = ""
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# Monkey-patch requests to skip SSL verification
old_request = requests.Session.request
def patched_request(self, *args, **kwargs):
    kwargs.setdefault("verify", False)
    return old_request(self, *args, **kwargs)
requests.Session.request = patched_request

# Suppress InsecureRequestWarning
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:
embedding = download_embeddings()

In [ ]:
vector= embedding.embed_query("Hakeem")

In [ ]:
vector

In [ ]:
print(len(vector))

In [ ]:
%pwd

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [ ]:
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

In [ ]:
import ssl
import urllib3

# Disable SSL verification globally for urllib3 (corporate proxy workaround)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Monkey-patch urllib3 HTTPSConnectionPool to disable SSL verification
_orig_urlopen = urllib3.HTTPSConnectionPool.urlopen
def _patched_urlopen(self, *args, **kwargs):
    self.cert_reqs = "CERT_NONE"
    self.ca_certs = None
    if not hasattr(self, '_ssl_context_patched'):
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        self.ssl_context = ctx
        self._ssl_context_patched = True
    return _orig_urlopen(self, *args, **kwargs)
urllib3.HTTPSConnectionPool.urlopen = _patched_urlopen

from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
pc

In [ ]:
from pinecone import ServerlessSpec
index_name="medical-chatbot"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")

    )


In [ ]:
index = pc.Index(index_name)

In [ ]:
'''from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embedding,
    index_name=index_name)'''

In [ ]:
# load existing code
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding 
)

ADDING MORE DATA TO EXCISTING DATABASE


In [ ]:
#dswidth=Document(page_content="Checking the vector database", metadata={"source": "test"})

In [ ]:
#docsearch.add_documents(documents=[dswidth])

In [ ]:
retriever=docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retrieved_docs = retriever.invoke("What is Acne")

In [ ]:
retrieved_docs

In [ ]:
from langchain_google_genai import GoogleGenerativeAI
chatmodel = GoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GEMINI_API_KEY)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.prompts import ChatPromptTemplate

In [ ]:
system_prompt=(
    "You are an Medical assistant for question-answering tasks."
    "Use the following pieces of retrieved context to answer"
    "the question.If you dont know the answer,say that you dont know."
    "use three sentences maximum to answer the question."
    "keep the answer concise and to the point."
    '\n\n'
    "{context}"
)

prompt=ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [ ]:
question_answering_chain=create_stuff_documents_chain(chatmodel,prompt)
rag_chain=create_retrieval_chain(retriever, question_answering_chain)

In [ ]:
response=rag_chain.invoke({"input":"What is Acne"})
print(response["answer"])

In [ ]:
response=rag_chain.invoke({"input":"What is Acne treatment?"})
print(response["answer"])